# Vision Project - Notebook 2
## Splitting the Dataset & Writing `data.yaml` for YOLO

> **Audience:** AI students and AI engineers continuing the passport-OCR project from notebook 1.

> **Goal:** Take the 50 annotated images + labels and split them into a clean `train/` and `val/` structure that YOLO can train on. Also write the `data.yaml` config that tells YOLO where everything lives.

---

### Where we are in the pipeline

```
Notebook 1      Annotation        ->  raw_data/images + raw_data/labels
Notebook 2 *    Data split + yaml ->  dataset/{images,labels}/{train,val} + data.yaml
Notebook 3      Training          ->  YOLO model weights
Notebook 4      Inference + OCR   ->  Structured passport JSON
```

---

# Part A - Quick Start (run these and you're done)

If you just need a working dataset split *right now*, run the next two cells in order. Detailed explanations of *why* every line exists are in **Part B** below.

### A.1 — Data split script

Run this from `ai/notebooks/`. It creates `dataset/images/{train,val}` and `dataset/labels/{train,val}`, then copies your annotated 50 images + their YOLO label files into the right folders with an 80/20 random split.

```
raw_data/
├── images/
├── labels/
```

```
dataset/
├── images/
│   ├── train/
│   └── val/
├── labels/
│   ├── train/
│   └── val/
```

In [7]:
import os
import shutil
import random
from pathlib import Path


IMAGES_DIR = "raw_data/images"
LABELS_DIR = "raw_data/labels"
DATASET_DIR = "dataset"

TRAIN_RATIO = 0.8
SEED = 42


# Create folder structure
def create_dataset_structure():
    folders = [
        "images/train",
        "images/val",
        "labels/train",
        "labels/val"
    ]

    for folder in folders:
        os.makedirs(os.path.join(DATASET_DIR, folder), exist_ok=True)

# Split dataset
def split_data():

    # Supported image formats
    image_extensions = (".jpg", ".jpeg", ".png")

    # Get all image files
    image_files = [
        f for f in os.listdir(IMAGES_DIR)
        if f.lower().endswith(image_extensions)
    ]

    # Check if images exist
    if len(image_files) == 0:
        print("No images found!")
        return

    # Shuffle images
    random.seed(SEED)
    random.shuffle(image_files)

    # Split index
    split_index = int(len(image_files) * TRAIN_RATIO)

    train_images = image_files[:split_index]
    val_images = image_files[split_index:]

    print(f"Total Images : {len(image_files)}")
    print(f"Train Images : {len(train_images)}")
    print(f"Val Images   : {len(val_images)}")

    # Copy train data
    for img_file in train_images:

        image_src = os.path.join(IMAGES_DIR, img_file)
        image_dst = os.path.join(DATASET_DIR, "images/train", img_file)

        shutil.copy2(image_src, image_dst)

        # Label file
        label_file = Path(img_file).stem + ".txt"

        label_src = os.path.join(LABELS_DIR, label_file)
        label_dst = os.path.join(DATASET_DIR, "labels/train", label_file)

        if os.path.exists(label_src):
            shutil.copy2(label_src, label_dst)
        else:
            print(f"Missing label: {label_file}")

    # Copy validation data
    for img_file in val_images:

        image_src = os.path.join(IMAGES_DIR, img_file)
        image_dst = os.path.join(DATASET_DIR, "images/val", img_file)

        shutil.copy2(image_src, image_dst)

        # Label file
        label_file = Path(img_file).stem + ".txt"

        label_src = os.path.join(LABELS_DIR, label_file)
        label_dst = os.path.join(DATASET_DIR, "labels/val", label_file)

        if os.path.exists(label_src):
            shutil.copy2(label_src, label_dst)
        else:
            print(f"Missing label: {label_file}")


create_dataset_structure()
split_data()

print("\nDataset split completed successfully.")

Total Images : 50
Train Images : 40
Val Images   : 10

Dataset split completed successfully.


### A.2 — Create `data.yaml`

This is the single config file that YOLO reads to know *where* your data lives and *what* it should detect. Drop it at `ai/notebooks/dataset/data.yaml`.

In [8]:
from pathlib import Path

DATA_YAML_CONTENT = '''path: /Users/Rizwan/Desktop/vision-project/ai/notebooks/dataset
# path:
# path: .

train: images/train
val: images/val

nc: 1
names: ['personal_no']
'''

yaml_path = Path("dataset/data.yaml")
yaml_path.parent.mkdir(parents=True, exist_ok=True)
yaml_path.write_text(DATA_YAML_CONTENT)

print(f"Wrote {yaml_path.resolve()}")
print("---")
print(yaml_path.read_text())

Wrote C:\Abdullah Ahmed\Ai course\Computer Vision\clause-ade\api\notebooks\dataset\data.yaml
---
path: /Users/Rizwan/Desktop/vision-project/ai/notebooks/dataset
# path:
# path: .

train: images/train
val: images/val

nc: 1
names: ['personal_no']



> ### Two things to know about that `data.yaml` before you train
>
> **1. The `path:` is hardcoded to one machine.**
> `/Users/Rizwan/Desktop/vision-project/ai/notebooks/dataset` only works on Rizwan's Mac. On any other machine, training will fail with "directory not found". On a teammate's laptop, on Colab, on a server — broken everywhere except one place.
>
> **Portable replacements** (uncomment the one that fits your setup):
> - `path: .` — dataset folder is the current working directory when you run training. Simple and works in most cases.
> - `path: ../dataset` — dataset is one folder up from where `data.yaml` lives. Useful when running from inside `dataset/` itself.
> - Just delete the `path:` key and use absolute paths in `train:` / `val:` — works but verbose.
>
> **2. The class config (`nc: 1`, `names: ['personal_no']`) means a *single-class* detector.**
> But in notebook 1 we labeled **5 classes** (`first_name, last_name, passport_no, nationality, dob`). These are two different valid strategies:
>
> | Strategy | `data.yaml` config | What the model learns |
> |---|---|---|
> | Single-region detection | `nc: 1` `names: ['personal_no']` | "Where is the whole passport in the image?" |
> | **Field-level detection (matches notebook 1)** | `nc: 5` `names: ['first_name', 'last_name', 'passport_no', 'nationality', 'dob']` | "Where are each of the 5 fields?" |
>
> If your `.txt` files in `raw_data/labels/` start with class IDs `0–4`, you must use the 5-class version below — otherwise YOLO will silently treat all of them as the single class and your training metrics will be nonsense. The cell below shows both options; pick the one that matches your annotations.

In [9]:
# # Optional: regenerate data.yaml to match the 5-class field-level setup from notebook 1.
# # Skip this cell if your annotations really are single-class 'pass'.

# from pathlib import Path

# FIELD_LEVEL_YAML = '''# Portable path — relative to wherever data.yaml lives
# path: .

# train: images/train
# val: images/val

# # 5 classes from notebook 1 annotation
# nc: 5
# names:
#   0: first_name
#   1: last_name
#   2: passport_no
#   3: nationality
#   4: dob
# '''

# # Uncomment to write the 5-class version instead:
# #
# # yaml_path = Path("dataset/data.yaml")
# # yaml_path.write_text(FIELD_LEVEL_YAML)
# # print(yaml_path.read_text())

# print("This cell is informational. Uncomment the 3 lines above if your annotations are 5-class.")

---

# Part B — Why everything above works (the real notebook)

Now that the data is split, let's understand *why* we did each step. This is what separates a copy-paste engineer from someone who can debug the inevitable problems.

## 1. What is data splitting?

**Data splitting** is the practice of dividing your full dataset into separate, non-overlapping subsets, where each subset has a different job during model development.

The fundamental idea is simple:

> **A model that has seen an image cannot honestly tell you whether it has *learned* anything from it.**
> Memorizing 50 passports and reciting them back is not learning. To know if the model actually understood the *pattern* of "what a passport field looks like", you must test it on images it has never seen.

That is the entire purpose of splitting.

## 2. Why bother splitting? Three concrete reasons

### Reason 1 — Detect overfitting

A neural network is a memorization machine. Give it 50 passports and enough capacity and it will memorize all 50 perfectly. Training accuracy = 100%. Looks great.

Now hand it the 51st passport — one it has never seen — and accuracy might drop to 40%. That gap (training vs. unseen) is **overfitting**. You can only see the gap if you keep some images aside as "unseen".

### Reason 2 — Honest evaluation

A model's reported accuracy is only meaningful if it was measured on data the model didn't train on. Training accuracy is a vanity metric — it tells you your *optimizer* worked, not that your *model* is good.

### Reason 3 — Fair model comparison

Should we use YOLOv8 nano or YOLOv11 small? Should we train at 640px or 1280px? Should we use 50 epochs or 100? Each of these is a decision, and you can only compare options fairly if you evaluate them on the same held-out data.

> Without splits, you have a model. With splits, you have **measurements** of a model. Measurements are what let you improve.

## 3. Train / Validation / Test — the three jobs

Production ML usually splits into **three** subsets, not two. Here's what each is for:

| Split | What it does | When the model "sees" it | Example size |
|---|---|---|---|
| **Train** | Model learns weights from these examples | Every epoch, every batch | 70–80% |
| **Validation** | Tune hyperparameters, pick best checkpoint, early stopping | After every epoch (read-only) | 10–20% |
| **Test** | Final, one-time honest report card | **Once**, at the end | 10–20% |

### Why the validation set is *not* the same as the test set

The validation set sounds like it's "unseen", and technically it is — the optimizer never updates weights on it. But you make decisions based on it: *"epoch 47 got 0.91 mAP on val, that's our best, ship it."* By picking that checkpoint, you've **leaked information from val into your model** — you've selected a model that happens to do well on val.

The test set fixes this. It is locked in a vault. You touch it once, at the very end, to get an honest number you can report. If you peek at it during development, it stops being a test set and becomes another val set.

> **Industry rule:** if you've ever made a single decision based on it, it isn't a test set anymore.

## 4. Why we skip the test set for our 50-image project

With only 50 images, splitting 70/15/15 gives:
- 35 train
- ~7 val
- ~7 test

7 images is too few for any metric to be statistically meaningful — change the random seed and your "test mAP" swings 20 points. So for this project we use **80/20 train/val** (40 train, 10 val) and accept that we can't ship a "final benchmark" with confidence.

When data is this precious, the industry-standard alternative is **k-fold cross-validation** — train 5 different models, each on a different 80/20 split, and report the average. We won't do that here, but here's how you'd think about it:

```
Fold 1:  TTTTTTTTV   ← V is val, T is train
Fold 2:  TTTTTTTVV
Fold 3:  TTTTTTVVT
Fold 4:  TTTTTVVTT
Fold 5:  TTTTVVTTT
                    → average mAP across all 5 folds = your real number
```

For 50 images this is what a careful researcher would do. For learning the basics — the single 80/20 split below is fine.

## 5. How to choose split ratios — quick reference

| Dataset size | Suggested split | Why |
|---|---|---|
| < 100 images (us) | 80/20 train/val, **no test** | Every image counts; consider k-fold |
| 100 – 1,000 | 70/15/15 | Standard "small dataset" recipe |
| 1,000 – 100,000 | 70/15/15 or 80/10/10 | Plenty of data for all three |
| > 100,000 | 98/1/1 | 1% of a million is still 10k — enough for stable metrics |

The intuition: as data grows, val/test percentages shrink, but their **absolute size** stays large enough to be reliable. Big-tech datasets like ImageNet (1.2M images) have only 50k val — about 4% — and that's plenty.

## 6. Splitting strategies — random isn't always right

Our script does a **random shuffle** split. That's the right default, but there are situations where it's *wrong* and silently destroys your model.

### Random split (what we use)
- Shuffle all images, take first 80% for train, rest for val.
- Good when: each image is independent, classes are roughly balanced.
- Our case: fine — passports were collected independently.

### Stratified split
- Ensures the **class distribution is identical** in train and val.
- Critical when classes are imbalanced. If 90% of your images are "no fraud" and 10% are "fraud", a random split might put almost all the fraud cases in val by chance.
- For our 5-class field detection, stratification is less critical because every image contains all 5 fields anyway.

### Group split (data leakage prevention)
- Critical when you have **multiple images of the same subject**. Imagine you photographed each of 25 passports twice (front-lit + side-lit) for 50 total images. A random split could put one shot in train and the other in val. The model essentially "sees" each passport during training, then "tests" on a near-duplicate. Inflated metrics, useless model.
- Solution: split by *passport ID*, not by image. All shots of passport #7 go to the same side.

### Time-based split
- For temporal data (financial, sensor, news). Past goes in train, future goes in val.
- Random split here would leak future information into training.

> **Question to ask yourself:** "If I picked an image at random from val, could the model have *seen something very similar* during training, just by being unlucky?" If yes, you need a smarter split than random.

## 7. The non-negotiables of any split

### 7.1 Reproducibility — always set a seed

```python
random.seed(SEED)        # SEED = 42 in our script
```

Without this, every time you run the script you get a different split. That means:
- Your "best model" could be best because it got an easy val set, not because it's actually good
- Teammates can't reproduce your results
- You can't tell whether an improvement came from your code change or from a different random draw

`SEED = 42` is the convention (Hitchhiker's Guide reference); any fixed integer works.

### 7.2 No data leakage

- Same image in train **and** val? Disaster — you're testing on training data.
- Augmented copies of one image scattered across both sets? Same disaster.
- Our script avoids this by partitioning the image list once and never overlapping.

### 7.3 Copy, don't move

Notice we use `shutil.copy2`, not `shutil.move`. **Always keep your raw data immutable.** If something goes wrong with the split (wrong ratio, wrong seed, missing files), you re-run with the originals untouched. Moving files means a buggy script can destroy your only copy.

> **Industry rule:** treat `raw_data/` like read-only.

### 7.4 Save the split manifest

For real projects, write the train/val assignment to a CSV alongside the data:
```csv
filename, split
img_001.jpg, train
img_002.jpg, val
...
```
Now anyone — your reviewer, your future self, your CI pipeline — can reproduce *exactly* the same split, even if the random seed implementation changes. We don't do this in our quick script, but for production you should.

## 8. Why YOLO needs `data.yaml` — the config file explained

YOLO (Ultralytics) is a **convention-based framework**. Instead of passing 10 command-line arguments every time you train, you write one YAML file describing your dataset, and YOLO reads everything from there.

### The training command becomes a one-liner

```bash
yolo detect train data=dataset/data.yaml model=yolov8n.pt epochs=100 imgsz=640
```

That single `data=dataset/data.yaml` argument tells YOLO:
- Where the images are
- Where the labels are (YOLO infers `images/train` ↔ `labels/train` automatically by replacing `images` with `labels` in the path)
- How many classes exist
- What each class is named (used in plots, logs, exported model metadata)

### Every field in `data.yaml` decoded

```yaml
path: /Users/Rizwan/Desktop/vision-project/dataset   # ROOT directory of dataset
train: images/train                                  # train images, relative to `path`
val:   images/val                                    # val images, relative to `path`
test:  images/test                                   # (optional) test images, also relative

nc: 1                                                # number of classes
names: ['personal_no']                                      # class names (index → name)
```

| Field | Required | What it does |
|---|---|---|
| `path` | optional | Root dataset directory. If omitted, `train`/`val` must be absolute. |
| `train` | Yes | Path (relative to `path`) to training images. YOLO auto-finds matching `.txt` labels by swapping `images` → `labels` in the path. |
| `val` | Yes | Same, for validation images. |
| `test` | optional | Used for the final test phase, not during training. |
| `nc` | Yes | Number of classes — must match the highest class ID in your `.txt` labels + 1. |
| `names` | Yes | List or dict mapping class index to human-readable name. Order matters: index 0 in your `.txt` files is `names[0]`. |

### Why YOLO infers `labels/` from `images/`

This is the magic line of the Ultralytics convention: when you say `train: images/train`, YOLO looks for label files at `labels/train/<same_filename>.txt`. **It does this by string substitution** — replacing `/images/` with `/labels/` in the path.

Practical consequence: your folder structure **must** look exactly like this, or labels won't be found:

```
dataset/
├── images/
│   ├── train/   img_001.jpg, img_002.jpg, …
│   └── val/     img_041.jpg, …
└── labels/
    ├── train/   img_001.txt, img_002.txt, …    ← same stems as images
    └── val/     img_041.txt, …
```

The split script we ran enforces exactly this layout. If you ever rename `images/` to `pictures/` or `labels/` to `annotations/`, YOLO's auto-discovery breaks silently and training proceeds on **zero labels** — every image treated as containing no objects. The model "trains" but learns nothing. Watch for this if you ever see "0 labels found" in the log.

### `nc` and `names` must agree with your `.txt` files

The number after `nc:` is **non-negotiable** — it must equal the count of unique class IDs in your label files. If your `.txt` files contain class IDs `0, 1, 2, 3, 4` but `nc: 1`, YOLO will throw an `AssertionError` at the start of training. (Better behaviour than silent failure, but still a stopper.)

`names` order must match the integer IDs you used during annotation:
- ID `0` in `.txt` → `names[0]`
- ID `1` in `.txt` → `names[1]`
- …

If you swap the order, the model still trains but every prediction comes back with the wrong label. This bug is a classic "it works but the answers are wrong" — easy to ship, hard to notice.

## 9. Walking through the split script, line by line

Now that the concepts are clear, here's why each chunk of the script exists.

### `IMAGES_DIR`, `LABELS_DIR`, `DATASET_DIR`

Constants at the top, in `UPPER_CASE`. Pulling these out means the next person can change paths in one place instead of hunting through the body of the script. Standard Python convention.

### `TRAIN_RATIO = 0.8` and `SEED = 42`

The two numbers that define your split. Putting them at the top, named, makes the script self-documenting — anyone reading it knows immediately "80/20, seeded".

### `create_dataset_structure()`

Creates `images/train`, `images/val`, `labels/train`, `labels/val` with `exist_ok=True` so re-running the script doesn't crash. **Idempotent operations are good operations.**

### Filtering image extensions

```python
image_files = [f for f in os.listdir(IMAGES_DIR)
               if f.lower().endswith(image_extensions)]
```

The `.lower()` is important — `IMG_001.JPG` and `img_001.jpg` should both pass. `os.listdir` returns hidden files and `.DS_Store` on macOS too, which would otherwise pollute your dataset.

### `random.seed(SEED)` *before* shuffle

The seed must be set **immediately before** the shuffle, not at module top, so that running the function twice gives the same split. If `random` is used elsewhere first (logging, hashing), the shuffle would consume different values.

### `shutil.copy2` not `shutil.copy`

`copy2` preserves file metadata (timestamps, permissions). Mostly cosmetic, but useful when debugging "when was this image added to my dataset?" later.

### `Path(img_file).stem + ".txt"`

Strips the extension cleanly: `passport_007.jpg` → `passport_007`. More robust than `img_file.replace('.jpg', '.txt')` which would fail on `.jpeg` or `.JPG`. **Always use `pathlib` for path manipulation.**

### Missing label warning, not crash

```python
if os.path.exists(label_src):
    shutil.copy2(...)
else:
    print(f"Missing label: {label_file}")
```

A missing label probably means you forgot to annotate one image. The script reports it but keeps going — you finish, see the warnings, fix them, re-run. Crashing on the first missing label would force you to fix-and-restart 50 times. **Be loud about problems but don't be brittle.**

## 10. Verify the split actually worked

Three sanity checks every dataset deserves. Run them all and look at the output before training.

In [10]:
# 10.1 — File counts in each split
from pathlib import Path

DATASET = Path("dataset")
splits = ["train", "val"]

for split in splits:
    n_imgs   = len(list((DATASET / "images" / split).glob("*")))
    n_labels = len(list((DATASET / "labels" / split).glob("*.txt")))
    print(f"{split:>5} | images: {n_imgs:>3} | labels: {n_labels:>3}")

train | images:  40 | labels:  40
  val | images:  10 | labels:  10


In [11]:
# 10.2 — Every image has a label, every label has an image
from pathlib import Path

DATASET = Path("dataset")

for split in ("train", "val"):
    img_stems = {p.stem for p in (DATASET / "images" / split).iterdir()
                 if p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
    lbl_stems = {p.stem for p in (DATASET / "labels" / split).glob("*.txt")}

    only_img = sorted(img_stems - lbl_stems)
    only_lbl = sorted(lbl_stems - img_stems)

    print(f"\n[{split}]")
    print(f"  Images without labels: {len(only_img)}")
    for s in only_img[:5]:
        print(f"    - {s}")
    print(f"  Labels without images: {len(only_lbl)}")
    for s in only_lbl[:5]:
        print(f"    - {s}")


[train]
  Images without labels: 0
  Labels without images: 0

[val]
  Images without labels: 0
  Labels without images: 0


In [12]:
# # 10.3 — Class distribution: do train and val have the same class mix?
# # (Run this only if your annotations are multi-class. For 1-class 'personal_no' it's trivially balanced.)

# from pathlib import Path
# from collections import Counter

# DATASET = Path("dataset")

# # Read class names from data.yaml (basic parser — for production use PyYAML)
# yaml_text = (DATASET / "data.yaml").read_text() if (DATASET / "data.yaml").exists() else ""

# def class_counts(split):
#     counter = Counter()
#     for txt in (DATASET / "labels" / split).glob("*.txt"):
#         for line in txt.read_text().strip().splitlines():
#             if line.strip():
#                 cls = int(line.split()[0])
#                 counter[cls] += 1
#     return counter

# train_counts = class_counts("train")
# val_counts   = class_counts("val")

# all_classes = sorted(set(train_counts) | set(val_counts))

# print(f"{'class':>5} | {'train':>6} | {'val':>5}")
# print("-" * 24)
# for c in all_classes:
#     print(f"{c:>5} | {train_counts.get(c, 0):>6} | {val_counts.get(c, 0):>5}")

# # Quick proportion check — train and val should have similar % per class
# train_total = sum(train_counts.values())
# val_total   = sum(val_counts.values())
# if train_total and val_total:
#     print(f"\n{'class':>5} | {'train %':>8} | {'val %':>7}")
#     print("-" * 28)
#     for c in all_classes:
#         tp = 100 * train_counts.get(c, 0) / train_total
#         vp = 100 * val_counts.get(c, 0) / val_total
#         print(f"{c:>5} | {tp:>7.1f}% | {vp:>6.1f}%")

### What to look for in the output above

- Cell 10.1: `train` should be ~40 images, `val` ~10 images. Image count == label count in each split.
- Cell 10.2: zero items in both lists. Any orphan = bug. Fix before training.
- Cell 10.3: per-class percentages should be **roughly equal** between train and val. If `passport_no` is 25% in train but 5% in val, your val set is unrepresentative — your reported metrics will be biased. With only 10 val images this can happen by random chance; if it does, consider re-running with a different `SEED` or implementing stratified split.

## 11. Industry-standard tips you'll wish you knew earlier

### 11.1 Treat `raw_data/` as immutable
Mount it read-only if you can. Every transformation produces a new directory. The original 50 images and original 50 labels never change — they are your ground truth.

### 11.2 Save the split manifest, not just the files
For real projects, after splitting, write a `splits.csv`:
```python
import csv
with open("dataset/splits.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["filename", "split"])
    for fn in train_images: w.writerow([fn, "train"])
    for fn in val_images:   w.writerow([fn, "val"])
```
Now the split is a *file*, not a side-effect of running a script. Reviewable, diff-able, version-controllable.

### 11.3 Consider stratified splits when class frequencies are unequal
```python
from sklearn.model_selection import train_test_split
train, val = train_test_split(
    files, test_size=0.2, random_state=42,
    stratify=[primary_class_of(f) for f in files],
)
```
For object detection, "primary class" is fuzzy (each image has multiple classes). A common workaround: stratify by *the rarest class present in each image*.

### 11.4 K-fold cross-validation when data is precious
```python
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(kf.split(files)):
    # train a separate model per fold, average final metrics
    ...
```
Five models cost 5× compute but give a far more reliable estimate of how your approach will perform on new data. For 50 images, this is genuinely worth doing if you care about the final number.

### 11.5 Hold out a true test set in production
Even if you don't use one in this notebook, in a real project: lock 10–15% of the data away on day 1. Never look at it. Never run inference on it. Touch it only when you're about to ship, to get an honest accuracy number for your stakeholders. Discipline here is what separates real ML teams from optimistic demos.

### 11.6 Version your dataset like code
- `dataset/v1/` — the 50-image split you trained your first model on
- `dataset/v2/` — same data + 50 more images you collected in week 3
- Tools: **DVC** (`dvc add dataset/`), **Roboflow** (versioned datasets), **Git LFS** for small datasets.
- When you compare model A (trained on v1) to model B (trained on v2), you need to be able to point to *exactly which images* each model saw. Otherwise comparison is meaningless.

### 11.7 Privacy stays paramount for documents
Add to `.gitignore`:
```gitignore
ai/notebooks/raw_data/
ai/notebooks/dataset/
*.pt
*.weights
runs/
```
Real passports are PII. Pushing them to GitHub — even in a private repo — is a data-handling violation in most jurisdictions.

### 11.8 Common sklearn helpers for non-CV projects
For tabular ML, you don't write split scripts — `sklearn.model_selection` has it covered:
- `train_test_split` — random or stratified
- `StratifiedKFold` — k-fold with class balance
- `GroupKFold` — k-fold respecting group boundaries (no leakage)
- `TimeSeriesSplit` — for temporal data

For image folders we wrote a custom script because we have to copy files, not just slice an array. The principles are identical.

## 12. Summary

### What we did

1. Created `dataset/{images,labels}/{train,val}/` directory structure
2. Randomly split 50 images 80/20 with a fixed seed
3. Copied (not moved) images and matching labels into the right folders
4. Wrote `data.yaml` with the YOLO config
5. Verified file counts, label-image matching, and class distribution

### What we learned

- **Why split:** to detect overfitting and get honest metrics
- **Train vs. val vs. test:** training, tuning, final report card — three jobs, three sets
- **Why no test set for us:** 50 images is too small; consider k-fold instead
- **Splitting strategies:** random, stratified, group, time-based — pick by data type
- **The non-negotiables:** seed for reproducibility, no leakage, copy don't move
- **`data.yaml`:** the YOLO convention that maps your folders + classes into a one-liner training command
- **The "images↔labels" auto-inference rule** that makes folder structure non-negotiable

### Coming next — `vision-v3.ipynb`

Train a YOLO model on this split:
- Pick a base model (YOLOv8n / YOLOv11n)
- Pick augmentations to compensate for our tiny 40-image training set
- Run training and read the metrics correctly
- Diagnose which class is performing worst and decide whether to label more data, augment more aggressively, or change strategy

> **Reminder:** with a 50-image dataset, no amount of clever modeling beats more annotated data. If you have time, the highest-leverage improvement is collecting and labeling 50 more passports. The model gets better the moment you press save in labelImg.